In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 48a974bf-6683-476d-a138-c9b5e4d7fa2c, 3, Finished, Available, Finished, False)

In [4]:
%%sql
Create schema LH_Silver.fact

StatementMeta(, 0da7485c-bd9d-424f-916a-a26e4d9e00e8, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
create table if not exists LH_Silver.dim.gold_Customer
(
  Customer_ID	string	,
Customer_Name	string	,
Segment	string	,
City	string	,
State	string	,
Country	string	,
Region	string	,
Created_TS	timestamp	,
Modified_TS	timestamp	  

)
using delta

StatementMeta(, 48a974bf-6683-476d-a138-c9b5e4d7fa2c, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
Max_Date=spark.sql("""select coalesce(max(Modified_TS),"1900-01-01") from LH_Silver.dim.gold_Customer""").first()[0]

StatementMeta(, 48a974bf-6683-476d-a138-c9b5e4d7fa2c, 5, Finished, Available, Finished, False)

In [4]:
Max_Date

StatementMeta(, 48a974bf-6683-476d-a138-c9b5e4d7fa2c, 6, Finished, Available, Finished, False)

'2026-09-17 03:58:53.147337'

In [8]:

df_bronze=spark.read.table("Lakehouse_Bronze.dbo.bronze_sales")

df_final=df_bronze.selectExpr("Customer_ID","Customer_Name","Segment","City","State", \
                   "Country","Region")\
                   .where(col("Modified_TS")>Max_Date)\
                   .drop_duplicates()

StatementMeta(, 0da7485c-bd9d-424f-916a-a26e4d9e00e8, 10, Finished, Available, Finished, False)

In [9]:
df_final.createOrReplaceTempView("ViewCustomer")

StatementMeta(, 0da7485c-bd9d-424f-916a-a26e4d9e00e8, 11, Finished, Available, Finished, False)

In [10]:
%%sql
Merge into LH_Silver.dim.gold_customer as gc
using ViewCustomer as vc
on gc.Customer_ID=vc.Customer_ID
when matched then 
update set
gc.Customer_Name	=	vc.Customer_Name	,
gc.Segment	=	vc.Segment	,
gc.City	=	vc.City	,
gc.State	=	vc.State	,
gc.Country	=	vc.Country	,
gc.Region	=	vc.Region	,
gc.Modified_TS	=	current_timestamp()	

when not matched then insert
(
gc.Customer_ID	,
gc.Customer_Name	,
gc.Segment	,
gc.City	,
gc.State	,
gc.Country	,
gc.Region	,
gc.Created_TS	,
gc.Modified_TS	
)
values
(
vc.Customer_ID	,
vc.Customer_Name	,
vc.Segment	,
vc.City	,
vc.State	,
vc.Country	,
vc.Region	,
current_timestamp(),
current_timestamp
)

StatementMeta(, 0da7485c-bd9d-424f-916a-a26e4d9e00e8, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>